In [1]:
import copy

def parse(fn):
    d = open(fn).read().split("\n\n")
    W,G = d
    W=[x.strip().split(": ") for x in W.split("\n")]
    W={x[0]: int(x[1]) for x in W}
    
    G=[x.split(" -> ") for x in G.split("\n")]
    G={b:a.split(" ") for a,b in G}
    return W,G


def getbit(i, v):
    return (v >> i) & 1



def compute(x, y, G):
    G = copy.deepcopy(G)
    W = {}
    
    last_l=None
    while len(G.keys()):
        l=len(G.keys())
        
        if l==last_l:
            #print("stuck")
            return None,[]
        last_l=l
        
        for k in G:
            a,o,b = G[k]
            
            _a = None
            if a.startswith("x"):
                _a = getbit(int(a[1:]), x)
            elif a.startswith("y"):
                _a = getbit(int(a[1:]), y)
            elif a in W:
                _a = W[a]
            
            _b = None
            if b.startswith("x"):
                _b = getbit(int(b[1:]), x)
            elif b.startswith("y"):
                _b = getbit(int(b[1:]), y)
            elif b in W:
                _b = W[b]
            
            if not _a is None and not _b is None:
                #print("found")
                if o == 'AND':
                    W[k]=_a and _b
                elif o == 'OR':
                    W[k]=_a or _b
                elif o == 'XOR':
                    W[k]=_a ^ _b
                else:
                    print(o)
                    assert(0)
                del G[k]
                break
        
        
    v = []
    i = 0
    while 1:
        n = "z%02d"%(i)
        #print(n)
        if n in W:
            v.append(W[n])
        else:
            break
        i+=1
    r = 0
    for b in reversed(v):
        r = r << 1
        r+=b
    
    hots=[]
    for k in W:
        if W[k] != 0:
            hots.append(k)
    return r, hots


def sources(n, G, depth=2):
    if n[0] in "xy":
        try:
            i = int(n[1:])
            return []
        except:
            pass
    
        
    r=[n]
    if n in G and depth>0:
        a,o,b=G[n]
        r+= sources(a, G,depth-1) + sources(b, G,depth-1)
    return r
    

    


def swap(a,b,G):
    G=copy.deepcopy(G)
    t=G[a]
    G[a]=G[b]
    G[b]=t
    return G
    
def test(s):
    
    G,W,bitfail=s
    G=copy.deepcopy(G)
    for b in range(45):
        #print(b)
        H=[] #hots
        E=[] #expected
        bv=1<<b
        back=1
        r,h=compute(bv,0,G)
        #print(r,bv)
        if r != bv:
            H+=h
            E+=sources("z%02d"%(b),G,back)
            #return b,H,E
        r,h=compute(0,bv,G)
        if r != bv:
            H+=h
            E+=sources("z%02d"%(b),G,back)
            #return b,H,E
        r,h=compute(bv,bv,G)
        if r != bv+bv:
            H+=h
            E+=sources("z%02d"%(b+1),G,back)
            #return b,H,E
        #print(H)
        if len(E):
            return (b,H,E)
        
    return None
        

def solve2(fn):
    W,G=parse(fn)
    #print(G)
    bit=0
    o=[(G,[],0)]
    c={}
    cnt=0
    while len(o):
        cnt+=1
        #print(len(o))
        o.sort(key=lambda x:x[2]-len(x[1])*6,reverse=True)
        s=o[0]
        o=o[1:]
        g,wires,bitfail=s
        #print(wires,bitfail)
        
        u=tuple(sorted(wires))
        if u in c:
            continue
        c[u]=1
        
        r=test(s)
        if r is None:
            #print("solution!",wires)
            return ",".join(sorted(wires))
        b,H,E=r
        print("%d fail bit:%d w:%d    "%(cnt,b,len(wires)),end="\r")
        for h in H:
            #print( [x for x in G.keys() ])
            for e in E:
                if h!=e and h not in wires and e not in wires:
                    if len(wires)<=6:
                        gg=swap(h,e,g)
                        w=wires+[h,e]
                        #print(w)
                        o.append((gg,w,b))
    print("\nend of line")
      

print("part1:", solve2("24.txt"), "\ndrg,gvw,jbp,jgc,qjb,z15,z22,z35")






part1: drg,gvw,jbp,jgc,qjb,z15,z22,z35 
drg,gvw,jbp,jgc,qjb,z15,z22,z35
